# Unidad 20 — Ámbito de variables y clausuras (closure)

Semana 2 · Clase 2 · Fuente: PDF parte 2, págs. 5–74

[Índice de la semana](../semana-02.ipynb) | [Anterior](../clase-1-funciones-recursividad-lambda/u19-lambda-filter-map-reduce.ipynb) | [Siguiente](u21-clases-y-objetos.ipynb)

**Nota sobre el PDF:** el PDF titula la unidad "Clousure" (con una "u" de más: la palabra correcta es *closure*, en español "clausura" o "cierre"). Las diapositivas de las págs. 71–74 ("Programación por parejas") llevan el rótulo "UNIDAD 21" en el encabezado, pero su portada (pág. 71) dice "Unidad 20", por lo que se incluyen aquí. La portada de la Unidad 21 empieza en la pág. 75.

In [1]:
def verificar(descripcion, obtenido, esperado):
    """Compara el resultado de un ejercicio con el esperado e informa sin detener el notebook."""
    if obtenido == esperado:
        print(f"Correcto: {descripcion}")
    else:
        print(f"Incorrecto: {descripcion}. Se esperaba {esperado!r} y se obtuvo {obtenido!r}")

## Objetivos de la unidad

Objetivos de aprendizaje que declara el PDF:

- Comprender y utilizar el ámbito de uso de las variables mediante la regla de ámbito (LEGB).
- Llamar y operar una función dentro de otra mediante funciones externas y funciones anidadas.
- Comprender el concepto de variable local dentro de una función y de variable global definida fuera de una función, y detectar la variable más cercana con la palabra clave `nonlocal` dentro de una función.
- Devolver la referencia de una función interna contenida en una función (clausura o *closure*).
- Construir una clausura utilizando una función lambda.
- Modificar una variable local de una clausura utilizando `nonlocal`.

**Palabras clave de la unidad:** regla de ámbito, función anidada, variable local y variable global, clausura (*closure*), `nonlocal`.

**Conceptos previos (según el PDF):** uso de funciones lambda; `filter`, `map` y `reduce` con lambda; paso de valores con parámetros variables, básicos y de palabra clave (Unidad 19 y anteriores).

**Referencia:** PDF parte 2, págs. 5–7

## Contexto

El PDF plantea un problema del mundo real: la **concentración urbana** causada por la industrialización. Algunos datos que menciona:

- Hacia 2018, cerca del 80 % de la población de Estados Unidos vivía en ciudades; se estima que el mundo llegará al 90 % en 2050 (fuente citada: informe *World Urbanization Prospects 2018*, de la ONU).
- La concentración excesiva agrava la contaminación atmosférica, la congestión del tráfico y la crisis inmobiliaria.
- La proporción de ciudades de más de 100.000 habitantes respecto de la población mundial era del 1,7 % en 1800 y del 13 % en 1950; en 1960 llegó al 20,1 %. En los últimos 150 años la población urbana aumentó 20 veces, mientras que la mundial solo 2,5 veces.

La **misión** de la unidad: analizar los datos de población de cuatro ciudades de una nación de Asia (A, B, C y D) con una función `urban` que imprima la población máxima, la mínima, la diferencia entre ambas y la media. Ese código usa variables **globales**, lo que conecta con el tema central de la unidad: el ámbito de las variables. Se resuelve en la sección *Proyecto de la unidad*.

**Referencia:** PDF parte 2, págs. 9–12

## 1. La regla de ámbito de Python (LEGB)

Cuando Python encuentra un nombre (una variable o una función), lo busca en este orden. Es la **regla de ámbito** (*scope rule*), conocida por sus siglas **LEGB**:

| Sigla | Nombre | Significado |
| --- | --- | --- |
| **L** | *Local* | Nombres definidos dentro de la función actual. |
| **E** | *Enclosing function local* | Nombres de una función que contiene a otra (ámbito envolvente). Python permite definir una función dentro de otra. |
| **G** | *Global* | Nombres del espacio de nombres del módulo (fuera de cualquier función). |
| **B** | *Built-in* | Nombres incorporados de Python (`print`, `len`, `abs`, ...). |

Si una variable se asigna dentro de una función, es **local**; si se asigna fuera de las funciones, es **global** a nivel de módulo. La búsqueda va de adentro hacia afuera y se detiene en el primer ámbito donde se encuentra el nombre.

**Ejemplo del PDF (pág. 18).** `y` es global, `x = 20` pertenece a `foo` y `a = 30` pertenece a `bar`. Desde el punto de vista de `bar`: `a` es L, `x` es E (viene de `foo`) e `y` es G.

In [2]:
x = 10
y = 11               # variable global (G)


def foo():
    x = 20           # variable local de foo; para bar es el ámbito envolvente (E)

    def bar():
        a = 30       # variable local (L) de bar
        print(a, x, y)   # a es L, x es E, y es G

    bar()
    x = 40           # se reasigna x en foo; bar verá el valor nuevo en la siguiente llamada
    bar()


foo()

30 20 11
30 40 11


La primera llamada a `bar()` imprime `30 20 11` y la segunda `30 40 11`: `bar` no guarda copia de `x`, la **busca** en el ámbito envolvente cada vez que se ejecuta.

**Nota sobre el PDF:** los comentarios de la diapositiva (`# la función foo corresponde a L`, `# la función de barra corresponde a E`) están mezclados por la traducción y no coinciden con el criterio correcto. La clasificación depende de **desde qué función se mire**: respecto de `bar`, `a` es L, `x` es E e `y` es G. El código y la salida (`30 20 11` / `30 40 11`) sí son correctos.

**Clave:** el nombre se resuelve en el orden L, E, G, B. El primer ámbito que contiene el nombre gana.

**Referencia:** PDF parte 2, págs. 17–18

### Cuidado con las funciones incorporadas (ámbito B)

Las funciones incorporadas viven en el último ámbito de la búsqueda. Si se crea una variable global con el mismo nombre, esta **bloquea** (*shadowing*) a la incorporada, porque G se busca antes que B. El PDF lo ilustra con `abs`: tras `abs = 10`, llamar a `abs(-5)` produce un `TypeError` porque `10` no es invocable. Al borrar la variable global con `del abs`, vuelve a verse la función incorporada.

In [3]:
print(abs)                 # función incorporada abs()
abs = 10                   # la variable global abs bloquea a la función incorporada
try:
    abs(-5)
except TypeError as error:
    print("TypeError:", error)

del abs                    # suprime la variable global; reaparece la función incorporada
print(abs(-10))

<built-in function abs>
TypeError: 'int' object is not callable
10


**Error común:** usar como nombre de variable un nombre incorporado (`abs`, `list`, `sum`, `max`, `min`, `len`, `str`, `input`...). El código sigue funcionando hasta que se intenta usar la función original y aparece un `TypeError` difícil de entender.

**Ampliación (no está en el PDF):** cómo ver qué nombres existen en cada ámbito. `locals()` muestra los de la función actual, `globals()` los del módulo y `dir(__builtins__)` los incorporados.

In [4]:
def mostrar_ambitos():
    variable_local = "soy local"
    print("Locales de mostrar_ambitos:", list(locals()))


mostrar_ambitos()
print("'abs' está entre los incorporados:", "abs" in dir(__builtins__))
print("'foo' está entre los globales:", "foo" in globals())

Locales de mostrar_ambitos: ['variable_local']
'abs' está entre los incorporados: True
'foo' está entre los globales: True


**Referencia:** PDF parte 2, pág. 19 (la sección de ampliación sobre `locals()`/`globals()` no figura en el PDF)

## 2. Variable local y variable global

Una variable creada dentro de una función es una **variable local** (L): se genera cuando se llama a la función y **desaparece de la memoria** al terminar la llamada. Una variable definida fuera de las funciones es **global** (G): permanece en memoria independientemente de las llamadas.

El PDF plantea la pregunta: si se asigna `counter = 200` dentro de la función y existe un `counter = 100` global, ¿se imprimirá 200 también fuera de la función? No. Ambas variables comparten **el mismo nombre pero tienen memoria separada**: la asignación dentro de la función crea una variable local nueva y no modifica la global.

In [5]:
def print_counter():
    counter = 200                          # crea una variable LOCAL nueva
    print('counter =', counter)            # valor del contador dentro de la función


counter = 100                              # variable global
print_counter()
print('counter =', counter)                # valor del contador fuera de la función

counter = 200
counter = 100


Salida: `counter = 200` (dentro) y `counter = 100` (fuera). La variable local `counter` desaparece cuando `print_counter` termina.

**Referencia:** PDF parte 2, págs. 20–22

### La palabra clave `global`

Para que la función use la variable global en lugar de crear una local, se declara `global counter` al inicio de la función. Con esa declaración, **cambiar el valor dentro de la función también cambia el valor fuera**.

In [6]:
def print_counter():
    global counter                         # declaración para usar la variable global counter
    counter = 200
    print('counter =', counter)            # valor del contador dentro de la función


counter = 100
print_counter()
print('counter =', counter)                # valor del contador fuera de la función

counter = 200
counter = 200


Ahora ambas salidas son `counter = 200`: la función modificó la variable global.

**Ampliación (no está en el PDF):** solo **leer** una variable global no requiere `global`; la declaración es necesaria cuando la función va a **asignarla** (si no, Python crea una local). Y si se lee y se asigna la misma variable sin declararla, ocurre un `UnboundLocalError`.

In [7]:
valor = 5


def solo_leer():
    return valor + 1            # leer una global no necesita 'global'


def leer_y_asignar():
    valor = valor + 1           # asigna 'valor': Python la trata como local y aún no tiene valor
    return valor


print(solo_leer())
try:
    leer_y_asignar()
except UnboundLocalError as error:
    print("UnboundLocalError:", error)

6
UnboundLocalError: cannot access local variable 'valor' where it is not associated with a value


**Clave:** sin `global`, asignar dentro de una función crea una variable local; con `global`, se modifica la variable del módulo.

**Error común:** creer que `counter = 200` dentro de una función cambia el `counter` global. Solo lo hace si se declaró `global counter`.

**Para el quiz:** una variable local desaparece al terminar la función; la palabra clave `global` hace que la función use la variable global en lugar de crear una local.

**Referencia:** PDF parte 2, págs. 21–24 (el ejemplo de `UnboundLocalError` es una ampliación)

## 3. Funciones de primera clase

En Python las funciones son **objetos de primera clase**: se tratan como cualquier otro objeto (números, cadenas, clases). Se pueden:

- pasar como argumento a otra función,
- asignar a una variable,
- devolver como valor de retorno,
- almacenar en estructuras de datos como listas o diccionarios.

**Ejemplo 1 (PDF pág. 25):** `callfunc` recibe una función como parámetro y la ejecuta.

In [8]:
def callfunc(func):
    func()


def greet():
    print('Hello')


print('llamada a función callfunc(greet)')
callfunc(greet)                  # se pasa la función greet (sin paréntesis) como argumento

llamada a función callfunc(greet)
Hello


**Nota sobre el PDF:** la salida de la diapositiva muestra `Hola`, pero el código imprime `'Hello'`; es un efecto de la traducción automática. La salida real es `Hello`.

**Ejemplo 2 (PDF pág. 26):** almacenar funciones en una lista y llamarlas por índice.

In [9]:
def plus(a, b):
    return a + b


def minus(a, b):
    return a - b


l_list = [plus, minus]
a = l_list[0](100, 200)          # llama a plus
b = l_list[1](100, 200)          # llama a minus
print('a =', a)
print('b =', b)

a = 300
b = -100


### Funciones de orden superior

Una **función de orden superior** (*higher-order function*) recibe funciones como argumentos o las devuelve. El PDF las presenta como "funciones de alto nivel implementadas por funciones de primera clase" (ya se usaron con `map`, `filter` y `reduce` en la Unidad 19).

**Ejemplo 3 (PDF pág. 27):** `f` recibe la función `g` y la aplica a `a` y `b`.

In [10]:
def add(a, b):
    return a + b


def f(g, a, b):
    return g(a, b)


print(f(add, 3, 4))

7


**Error común:** escribir `callfunc(greet())` en lugar de `callfunc(greet)`. Con paréntesis se **ejecuta** `greet` y se pasa su resultado (`None`), no la función.

**Para el quiz:** pasar una función como argumento se escribe sin paréntesis; los paréntesis la ejecutan.

**Referencia:** PDF parte 2, págs. 25–27

## 4. Función externa y función anidada

Python permite **definir una función dentro de otra**. La función interna se llama **función anidada** (*nested function*) y la que la contiene, función externa. Además, una función puede devolver otra función.

**Ejemplo (PDF pág. 28):** `decorate` define dos funciones internas (`italic` y `bold`) y devuelve una u otra según el parámetro `style`. Así se evita repetir sentencias de control en cada uso.

In [11]:
def decorate(style='italic'):
    def italic(s):
        return '<i>' + s + '</i>'

    def bold(s):
        return '<b>' + s + '</b>'

    if style == 'italic':
        return italic
    else:
        return bold


dec = decorate()
print(dec('hello'))
dec2 = decorate('bold')
print(dec2('hello'))

<i>hello</i>
<b>hello</b>


**Nota sobre el PDF:** el texto de la diapositiva dice "función cursiva, negrita en la función decorar" porque tradujo `italic` (cursiva) y `bold` (negrita) y `decorate` (decorar). Los identificadores reales son `italic`, `bold` y `decorate`.

### Razones para usar funciones anidadas

- Una función anidada puede **leer libremente las variables de la función padre**, a diferencia de las funciones definidas afuera.
- Mejora la legibilidad. Solo por legibilidad también se puede declarar la función auxiliar fuera, como en el siguiente ejemplo del PDF (pág. 29), que funciona igual.
- La razón más importante: la función anidada es la base del concepto de **clausura**, que se verá más adelante. Una de las condiciones para tener una clausura es ejecutar la función anidada fuera de su contexto original.

In [12]:
def another_func():
    print("hello")


def outer_func():
    return another_func()


outer_func()

hello


**Clave:** una función anidada solo existe dentro de la función externa y puede usar sus variables (ámbito E).

**Para el quiz:** la razón principal para usar funciones anidadas es entender y construir clausuras.

**Referencia:** PDF parte 2, págs. 28–30

## 5. Proceso de búsqueda de la variable: `global` y `nonlocal`

### 5.1 `global` desde una función anidada

Si una función anidada necesita **modificar** una variable global, se declara `global`. Significa "usaré la variable global, no la del ámbito actual". La declaración `global` busca la variable en el espacio de nombres del módulo, saltándose los ámbitos intermedios.

**Ejemplo (PDF pág. 31):**

In [13]:
n1 = 1


def func1():
    def func2():
        global n1          # n1 se busca en el espacio de nombres global
        n1 += 1
        print(n1)          # imprime 2

    func2()


func1()

2


### 5.2 Necesidad de `nonlocal`

Si la variable **no es global** sino que vive en una función envolvente, `global` no sirve: Python busca una variable global con ese nombre y no la encuentra. El PDF muestra este `NameError` (el título de la diapositiva, "ErrorNombre", es la traducción de `NameError`).

In [14]:
def func1():
    n2 = 1

    def func2():
        global n2          # n2 no existe como variable global
        n2 += 1            # aquí falla: no hay un n2 global que incrementar
        print(n2)

    func2()


try:
    func1()
except NameError as error:
    print("NameError:", error)

NameError: name 'n2' is not defined


La solución es usar **`nonlocal`**: indica que la variable no es local ni global, y que debe enlazarse con la **variable más cercana** de las funciones envolventes. Ese enlace con la variable más cercana se denomina **vinculación** (*binding*).

**Nota sobre el PDF:** la diapositiva comenta `# error` en la línea del `print(n2)`, pero el fallo ocurre antes, en `n2 += 1`, porque ahí se intenta leer un `n2` global inexistente. El mensaje (`name 'n2' is not defined`) coincide con el del PDF.

**Ejemplo corregido con `nonlocal` (PDF pág. 33):**

In [15]:
def func1():
    n3 = 1

    def func2():
        nonlocal n3        # n3 no es local ni global: se vincula con la n3 de func1
        n3 += 1
        print(n3)          # No es error: imprime 2

    func2()


func1()

2


### 5.3 `nonlocal` y vinculación (PDF pág. 34)

`nonlocal` solo puede enlazar con una variable que se encuentra **en una función exterior** (un nivel fuera de la función que lo usa). En el ejemplo, `x = 20` es global, pero `nonlocal x` dentro de `g` se vincula con la `x = 40` de `f`, que es la más cercana.

In [16]:
x = 20                    # variable global


def f():
    x = 40

    def g():
        nonlocal x        # se vincula con la x de f, no con la global
        x = 80

    g()
    print(x)              # 80: g cambió la x de f


f()
print(x)                  # 20: la x global no cambió

80
20


Explicación de las líneas 7, 8 y 11 del PDF: se ejecuta `g` (que aplica `nonlocal`), se imprime la `x` de `f` (que `g` cambió a 80) y, tras ejecutar todo, se imprime la `x` global, que sigue valiendo el valor original 20.

### 5.4 `nonlocal` no puede enlazar con una variable global (PDF pág. 35)

Si se declara `nonlocal x` en una función que **no está anidada** en otra, no hay ninguna variable envolvente a la que vincularse (la global no cuenta) y Python lanza un `SyntaxError` **al compilar**, antes de ejecutar nada. Como el error es de sintaxis, para mostrarlo sin romper el notebook se compila el texto del código dentro de un `try`.

In [17]:
codigo_con_error = """
x = 70  # variable global
def f():
    nonlocal x
    x = 140

f()
print(x)
"""

try:
    compile(codigo_con_error, "<ejemplo>", "exec")
except SyntaxError as error:
    print("SyntaxError:", error.msg)

SyntaxError: no binding for nonlocal 'x' found


### 5.5 Proceso de búsqueda de `nonlocal` (PDF pág. 36)

`nonlocal` enlaza con la variable del **espacio de nombres envolvente más cercano**. En el ejemplo hay tres niveles: `f` define `a = 777`, `g` define `a = 100` y `h` declara `nonlocal a`. Como `g` está más cerca de `h` que `f`, `h` modifica la `a` de `g` y la `a` de `f` queda intacta.

In [18]:
def f():
    a = 777

    def g():
        a = 100

        def h():
            nonlocal a      # la variable más cercana es la a de g
            a = 333

        h()
        print("[Level 2] a = {}".format(a))   # 333: h cambió la a de g

    g()
    print("[Level 1] a = {}".format(a))       # 777: la a de f no cambió


f()

[Level 2] a = 333
[Level 1] a = 777


**Clave:**
- `global` enlaza con la variable del módulo; `nonlocal` enlaza con la variable de la función envolvente **más cercana**.
- `nonlocal` exige que exista esa variable en alguna función exterior; con una variable solo global produce `SyntaxError`.

**Error común:** usar `global` para una variable que vive en la función exterior (da `NameError`) o usar `nonlocal` para una variable global (da `SyntaxError`).

**Para el quiz:** `nonlocal` busca la variable del espacio de nombres envolvente más cercano; esa conexión se llama vinculación.

**Referencia:** PDF parte 2, págs. 31–36

## 6. Concepto de clausura (*closure*)

La forma básica de una clausura: se **crea una función anidada dentro de otra y se devuelve la función anidada**. La función interna "queda cerrada" con las variables de la función externa: aunque la función externa ya terminó, esas variables **no desaparecen de la memoria** y la función interna puede seguir usándolas en llamadas posteriores.

El PDF introduce el término **variable libre**: es una variable que se usa en un bloque de código pero no es global ni está definida dentro de ese bloque; es un concepto relativo. Desde el punto de vista de `mult`, `a` es una variable libre; para `closure_calc`, `a` es solo una variable local.

**Ejemplo (PDF pág. 37):** `closure_calc` devuelve la función `mult`, que recuerda `a = 2`.

In [19]:
def closure_calc():
    a = 2

    def mult(x):
        return a * x

    return mult


c = closure_calc()        # c ahora referencia a la función mult
print(c(1), c(2), c(3))

2 4 6


**Nota sobre el PDF:** en la pág. 37 la línea 7 aparece como `c = closure_calc():` con dos puntos al final, lo que sería un `SyntaxError`; el código correcto no lleva los dos puntos (como se ejecutó arriba). Además, el nombre de la función varía entre diapositivas (`closure_calc`, `cierre_calculo`, `clouser_calc`) por la traducción y por errores de escritura; es la misma función.

### Análisis de la clausura (PDF pág. 38)

El valor devuelto por `closure_calc` es la función `mult`. Como las funciones son de primera clase, se puede llamar a `mult` a través de `c`. Aunque `closure_calc` ya terminó, `mult` sigue funcionando porque la variable `a` **no desapareció**: el área encerrada por `closure_calc` (la función `mult` junto con `a = 2`) es la clausura.

**Ampliación (no está en el PDF):** Python permite inspeccionar qué variables libres captura una función.

In [20]:
print("Variables libres de c:", c.__code__.co_freevars)
print("Valor capturado de a:", c.__closure__[0].cell_contents)

Variables libres de c: ('a',)
Valor capturado de a: 2


### Finalidad de la clausura (PDF pág. 39)

- En la programación tradicional había que declarar una variable **global** para usarla en varias funciones sin depender de que una función terminara.
- Generar variables globales sin control provoca **efectos secundarios** que dificultan la depuración.
- Para crear una clausura se define la variable que se desea conservar y la función que la usa, y se encierra todo dentro de otra función. Si esa función externa devuelve la función interna, la función externa se convierte en una clausura.
- Así se **reduce el uso de variables globales** y se **oculta** al exterior el funcionamiento interno.

**Nota sobre el PDF:** la diapositiva dice "reducir la frecuencia de la variable general"; es una traducción poco feliz de "reducir el uso de variables globales".

### Usos principales de la clausura (PDF págs. 40–41)

1. Limitar el uso de variables globales.
2. Ocultación de datos (*data hiding*): el dato queda como variable local de la función que lo encierra y solo se accede mediante la función devuelta.

Una clausura proporciona un **espacio de nombres independiente** para cada función creada, de modo que se puede definir una función que se comporte de forma independiente, como una **instancia** de un objeto (las clases se ven en la Unidad 21). En cuanto a la eficiencia, el PDF indica que la clausura es poco eficiente en memoria, aunque a veces puede ser más eficaz porque reduce las variables globales.

**Ejemplo (PDF pág. 41):** `makecounter` crea un contador privado. `c1` y `c2` son clausuras distintas, cada una con **su propia** variable `count`. Aquí `nonlocal` es indispensable, porque `counter` modifica `count`, que vive en la función externa.

In [21]:
def makecounter():
    count = 0

    def counter():
        nonlocal count       # count vive en makecounter, no en counter
        count += 1
        return count

    return counter


c1 = makecounter()
c2 = makecounter()
print('c1', c1())
print('c1', c1())
print('c2', c2())

c1 1
c1 2
c2 1


La salida es `c1 1`, `c1 2`, `c2 1`: el contador de `c1` y el de `c2` se mantienen en espacios de memoria separados. Y no existe ninguna variable global `count` que alguien pueda modificar por accidente (ocultación de datos).

**Error común (ampliación, no está en el PDF):** olvidar `nonlocal` al modificar la variable capturada. Leer una variable libre funciona sin declararla, pero **asignarla** convierte el nombre en local y falla con `UnboundLocalError`.

In [22]:
def makecounter_malo():
    count = 0

    def counter():
        count += 1           # sin nonlocal: Python cree que count es local
        return count

    return counter


contador_malo = makecounter_malo()
try:
    contador_malo()
except UnboundLocalError as error:
    print("UnboundLocalError:", error)

UnboundLocalError: cannot access local variable 'count' where it is not associated with a value


### Cómo generar una clausura (PDF pág. 42)

Se genera escribiendo una función anidada y devolviéndola. En el ejemplo, `mul_add` usa las variables `a` y `b` de `calc` (líneas 5 y 6 del PDF: calcula con ellas y devuelve la función `mul_add`).

In [23]:
def calc():
    a = 3
    b = 5

    def mul_add(x):
        return a * x + b

    return mul_add


c = calc()
print(c(1), c(2), c(3), c(4), c(5))

8 11 14 17 20


**Clave:**
- Clausura = función anidada que se devuelve y que conserva las variables de la función externa.
- Sirve para ocultar datos y para evitar variables globales.
- Cada llamada a la función externa crea una clausura **independiente** con sus propias variables.

**Para el quiz:** una función es una clausura cuando una función anidada usa variables de la función externa y esa función anidada se devuelve; los datos quedan ocultos y no son globales.

**Referencia:** PDF parte 2, págs. 37–42

## 7. Clausura con lambda

Una función lambda es una **función anónima**; una clausura es una función que se reutiliza **después de conservar el entorno** que la envuelve. Suelen usarse juntas y es fácil confundirlas, pero son conceptos distintos. Con lambda la clausura se escribe de forma más simple: la función externa devuelve directamente la expresión `lambda x: a * x + b`.

**Ejemplo (PDF pág. 43):**

In [24]:
def clouser_calc():
    a = 2
    b = 3
    return lambda x: a * x + b       # making closure with lambda


c = clouser_calc()
print(c(1), c(2), c(3), c(4), c(5))  # imprime 5 7 9 11 13

5 7 9 11 13


**Comparación antes y después de usar lambda (PDF pág. 44):** el mismo `calc` con `a = 3` y `b = 5` se puede escribir con `def` interna o con lambda, y el resultado es idéntico (`8 11 14 17 20`).

In [25]:
def calc_def():
    a = 3
    b = 5

    def mul_add(x):
        return a * x + b

    return mul_add


def calc_lambda():
    a = 3
    b = 5
    return lambda x: a * x + b       # expresión más sencilla


c_def = calc_def()
c_lambda = calc_lambda()
print(c_def(1), c_def(2), c_def(3), c_def(4), c_def(5))
print(c_lambda(1), c_lambda(2), c_lambda(3), c_lambda(4), c_lambda(5))

8 11 14 17 20
8 11 14 17 20


**Nota sobre el PDF:** el ejemplo de la pág. 43 usa `a = 2`, `b = 3` (salida `5 7 9 11 13`), mientras que la comparación de la pág. 44 usa `a = 3`, `b = 5` (salida `8 11 14 17 20`). Son dos ejemplos distintos, no hay inconsistencia.

**Clave:** una lambda devuelta por una función **es** una clausura si usa variables de la función externa (`a` y `b`).

**Referencia:** PDF parte 2, págs. 43–44

## 8. Modificar una variable local de una clausura

Para **cambiar** (no solo leer) la variable local que una clausura conserva, se usa `nonlocal`. Así la clausura puede mantener un estado que evoluciona con cada llamada.

**Ejemplo (PDF pág. 45):** `mult_add` acumula en `total` los resultados de `a * x + b` en cada llamada. `total` es la variable local de la clausura y `nonlocal` la vincula con la variable de `calc`.

In [26]:
def calc():
    a = 2
    b = 3
    total = 0

    def mult_add(x):
        nonlocal total                  # total pertenece a calc: se modifica desde la clausura
        total = total + a * x + b
        return total

    return mult_add


c = calc()
print(c(1), c(2), c(3))

5 12 21


Cálculo paso a paso: `c(1)` suma `2*1+3 = 5` (total 5); `c(2)` suma `2*2+3 = 7` (total 12); `c(3)` suma `2*3+3 = 9` (total 21). La salida es `5 12 21`.

**Clave:** `nonlocal` convierte a la clausura en una función **con memoria**: el estado persiste entre llamadas.

**Para el quiz:** para modificar una variable local de la función externa desde la función interna (clausura) se usa `nonlocal`.

**Referencia:** PDF parte 2, pág. 45

## 9. Codificación en papel (ejercicios del PDF)

El PDF propone tres ejercicios de "codificación en papel" (escribir el código completo y la salida esperada en la nota; 5 minutos cada uno). Se resuelven en la sección *Práctica* (ejercicios 1 a 3):

- **Q1 (pág. 47):** crear una función anidada: una función `saludos` que contiene a `decir_hi` (que muestra `hola`), llamar a `decir_hi` dentro de `saludos` y luego llamar a `saludos`. Condición de ejecución: la salida esperada es `hello`.
- **Q2 (pág. 48):** escribir la función `calc` (con `a = 3`, `b = 5` y la función interna `mul_add(x)` que devuelve `a * x + b`), asignar `num = calc()` y ejecutar `num(3)`; el resultado debe ser `14`.
- **Q3 (pág. 49):** construir `mul_add` con una expresión lambda e imprimir `14`.

**Nota sobre el PDF:** en Q1 la traducción mezcló los nombres: el enunciado habla de `saludos` / `decir_hi` y de `greetings` / `say_hi` (son la misma pareja en español e inglés), y pide imprimir `'hola'` mientras que la condición de ejecución muestra `hello`. En Q2 aparece `devolver` en lugar de `return`, que sería un `SyntaxError`; además "asigna `calc` a la variable `num`" debe leerse como `num = calc()` (se asigna el **resultado** de llamar a `calc`, es decir, la clausura).

**Referencia:** PDF parte 2, págs. 46–49

## 10. Gráficos de tortuga (turtle)

Esta parte del PDF (que figura dentro de la unidad aunque no está en el encargo original) introduce el módulo `turtle`, una herramienta de dibujo incluida en Python. Permite dibujar gráficos en una ventana: un cursor (la "tortuga") se mueve y dibuja líneas.

**Ejemplo del PDF (págs. 52–53):**

```python
import turtle as t

t.setup(width=400, height=400)   # tamaño de la ventana
for i in range(200):
    t.forward(i)                 # avanza i píxeles dibujando una línea
    t.left(93)                   # gira 93 grados a la izquierda
t.done()                         # mantiene la ventana abierta
```

Este código **no se ejecuta en este notebook** porque abre una ventana gráfica aparte; para probarlo, guárdalo en un archivo `.py` y ejecútalo desde una terminal. Con cada iteración la línea es más larga y el giro de 93 grados genera un patrón en espiral.

**Nota sobre el PDF:** la diapositiva (pág. 53) dice "`t.forward(i)` mueve el cursor hacia la izquierda" y "`t.left(93)` gira el cursor 98 grados". Ambas frases son erróneas: `forward` mueve el cursor **hacia adelante** (en la dirección en que apunta) y `left(93)` gira **93** grados a la izquierda, como indica el código.

### Comandos de tortuga (PDF págs. 54–55)

La traducción automática tradujo los nombres de los comandos. La tabla muestra el nombre real que se escribe en el código:

| Comando real | Tarea |
| --- | --- |
| `begin_fill()` ... `end_fill()` | Colorea el área dibujada entre ambas llamadas. |
| `color(c)` | Cambia el color (por ejemplo `'red'`, `'green'`, `'blue'`, `'black'`, `'gray'`, `'pink'`); el PDF lo muestra traducido. |
| `shape(s)` | Cambia la forma de la tortuga: `'arrow'`, `'turtle'`, `'circle'`, `'square'`, `'triangle'`, `'classic'`. |
| `shapesize(s)`, `shapesize(w, h)` | Modifica el tamaño de la tortuga. |
| `pos()`, `position()` | Devuelve la posición actual. |
| `xcor()`, `ycor()` | Devuelven la coordenada x o y actual. |
| `heading()` | Devuelve la orientación actual (el PDF dice "encabezamiento"). |
| `distance(x, y)` | Distancia entre la posición actual y la indicada. |
| `penup()`, `pu()`, `up()` | Levanta el lápiz (deja de dibujar). |
| `pendown()`, `pd()`, `down()` | Baja el lápiz (vuelve a dibujar). |
| `pensize(w)`, `width(w)` | Cambia el grosor del trazo. |
| `circle(r)` | Dibuja un círculo de radio `r` en la posición actual. |
| `goto(x, y)`, `setpos(x, y)`, `setposition(x, y)` | Envía la tortuga a una coordenada; dibuja una línea solo si el lápiz está bajado. |
| `stamp()` | Deja una copia del tamaño, color y forma de la tortuga en la posición actual. |
| `home()` | Devuelve la tortuga a su posición y dirección iniciales. |
| `textinput(titulo, mensaje)` | Abre una ventana para pedir texto al usuario; se usa como `turtle.textinput()`. |

**Nota sobre el PDF:** en la tabla, los nombres de los comandos aparecen en español (`forma`, `posición`, `encabezamiento`, `distancia`, `tamaño`, `anchura`, `círculo`, `sello`, `inicio`) por la traducción automática; esos nombres **no existen** en `turtle`. Además `textinput` "recibe picaduras" es una traducción de "recibe *strings*" (cadenas de texto). Un ejemplo correcto, en una terminal, dibujando un cuadrado relleno:

```python
import turtle as t

t.color('blue')
t.begin_fill()
for _ in range(4):
    t.forward(100)
    t.left(90)
t.end_fill()
t.done()
```

**Ampliación (no está en el PDF):** el ejemplo del cuadrado es propio y se presenta solo como ilustración de los comandos de la tabla.

**Referencia:** PDF parte 2, págs. 51–55

## 11. Expresiones regulares y metacaracteres

Las **expresiones regulares** son un lenguaje para describir **patrones** que deben cumplir las cadenas de texto. Son importantes para el análisis y extracción (*scraping* o rastreo) de páginas web: la extracción de datos de páginas web se denomina rastreo. En Python se usan con el módulo `re`. El PDF solo trata las funciones básicas de búsqueda.

### 11.1 Búsqueda con `re.search`

`re.search(patron, cadena)` busca el patrón en la cadena. Si hay coincidencia devuelve un objeto `re.Match`; si no, devuelve `None`. En el ejemplo se busca `Life` (con L mayúscula) en dos cadenas.

In [27]:
import re

txt1 = "Life is too short, you need python."
txt2 = "The best moments of my life."
print(re.search('Life', txt1))   # comprueba si Life está en la frase
print(re.search('Life', txt2))   # no está con L mayúscula: devuelve None

<re.Match object; span=(0, 4), match='Life'>
None


`txt1` contiene `Life` con L mayúscula, por eso devuelve el objeto de coincidencia; `txt2` solo contiene `life` en minúscula, por eso devuelve `None`.

**Nota sobre el PDF:** el texto traducido dice "Vida" donde el código dice `Life`/`life`; el patrón real es la palabra inglesa de los ejemplos.

El objeto `Match` ofrece los métodos `group()` (la parte coincidente), `start()` y `end()` (posiciones de inicio y fin) y `span()` (la tupla `(inicio, fin)`). Con `span` se puede recuperar la coincidencia con un corte de cadena.

In [28]:
match = re.search('Life', txt1)
print(match.group())      # texto que coincidió
print(match.start())      # posición inicial
print(match.end())        # posición final
print(match.span())       # (inicio, fin)
print(txt1[0:4])          # el corte con el span devuelve la coincidencia

Life
0
4
(0, 4)
Life


### 11.2 Compilar un patrón y usar grupos

`re.compile(patron)` crea un objeto de expresión regular reutilizable (aquí llamado `regex`). Dentro del patrón, `\d{3}` encuentra tres dígitos y `\d{4}` cuatro dígitos; los **paréntesis** forman **grupos** numerados desde 1, que se leen con `group(1)`, `group(2)`.

**Ejemplo del PDF (pág. 60):** extraer un número de teléfono y sus partes.

In [29]:
text = 'please call 010-2345-1234'
regex = re.compile(r'(\d{3})-(\d{4}-\d{4})')   # dos grupos: 3 dígitos y 4-4 dígitos
match_obj = regex.search(text)
print(match_obj.group())     # toda la coincidencia
print(match_obj.group(1))    # primer grupo
print(match_obj.group(2))    # segundo grupo

010-2345-1234
010
2345-1234


**Ampliación (no está en el PDF):** conviene escribir los patrones como cadenas "crudas" (`r'...'`), para que Python no interprete las barras invertidas (`\d`, `\s`, ...) como caracteres de escape de la propia cadena.

**Nota sobre el PDF:** en la diapositiva ambos grupos aparecen rotulados "grupo(1)" en las flechas; el segundo es `group(2)`, como muestra el código y su salida (`2345-1234`).

### 11.3 Alternativa con `|` y clases de caracteres con `[]`

`re.search('Life', txt2)` devuelve `None` porque `txt2` contiene `life` con l minúscula. Hay dos formas de aceptar ambas variantes:

- **Alternativa `|`:** `'Life|life'` significa "la cadena `Life` **o** la cadena `life`". No significa `Vida|vida` mezclando letras: separa dos opciones completas.
- **Clase de caracteres `[]`:** `'[Ll]ife'` significa "una `L` o una `l` seguida de `ife`". Los corchetes expresan un conjunto o rango de caracteres; por ejemplo `[0-9]` son todos los dígitos del 0 al 9.

Los caracteres con significado especial como `[]`, `-` y `|` se llaman **metacaracteres**.

In [30]:
print(re.search('Life|life', txt2))     # alternativa: Life o life
print(re.search('[Ll]ife', txt2))       # clase de caracteres: L o l seguida de ife

<re.Match object; span=(23, 27), match='life'>
<re.Match object; span=(23, 27), match='life'>


En `txt2` la coincidencia `life` está en las posiciones `(23, 27)`.

### 11.4 Anclas `^` y `$`

- `^` coincide con el **inicio** de la cadena: `'^Life'` solo coincide si la cadena **empieza** por `Life`.
- `$` coincide con el **final** de la cadena: `'life$'` solo coincide si la cadena **termina** en `life`.

In [31]:
txt1 = "Life is too short, you need python."
txt2 = "The best moments of my life."
txt3 = "My Life My Choice."
print(re.search('^Life', txt1))    # empieza por Life: coincide
print(re.search('^Life', txt2))    # no empieza por Life: None
print(re.search('^Life', txt3))    # Life está pero no al principio: None

<re.Match object; span=(0, 4), match='Life'>
None
None


In [32]:
txt1 = 'Who are you to judge the life I live'
txt2 = 'The best moments of my life'
print(re.search('life$', txt1))     # no termina en life: None
print(re.search('life$', txt2))     # termina en life: coincide

None
<re.Match object; span=(23, 27), match='life'>


**Nota sobre el PDF:** la pág. 64 dice "en la segunda fila de la tabla" refiriéndose a la tabla de metacaracteres de la pág. 65 (la fila de `$`); el orden de las diapositivas no coincide con esa referencia.

### 11.5 Metacaracteres esenciales (PDF pág. 65)

| Expresión | Función | Descripción |
| --- | --- | --- |
| `^` | inicio | Coincide con el inicio de la cadena. |
| `$` | fin | Coincide con el final de la cadena. |
| `.` | carácter | Coincide con un carácter cualquiera. |
| `\d` | número | Coincide con un dígito. |
| `\w` | carácter o número | Coincide con una letra, un dígito o el guion bajo. |
| `\s` | espacio en blanco | Coincide con un espacio, un tabulador, un retorno de carro o un salto de línea. |
| `\S` | no espacio en blanco | Todos los caracteres excepto los espacios en blanco. |
| `*` | repetición | 0 o más repeticiones del elemento anterior. |
| `+` | repetición | 1 o más repeticiones del elemento anterior. |
| `[abc]` | rango de caracteres | Coincide con `a`, `b` o `c`. |
| `[^abc]` | rango negado | Cualquier carácter que **no** sea `a`, `b` ni `c`. |

El PDF destaca como los más importantes el **punto** (`.`: cualquier carácter) y el **asterisco** (`*`: cualquier cantidad de repeticiones).

**Ampliación (no está en el PDF):** el PDF define `\w` como "carácter o número"; en rigor incluye también el guion bajo. Además, el punto no coincide con el salto de línea, salvo que se use una opción especial.

**El punto `.`** (PDF págs. 66): `'A..A'` busca una `A`, dos caracteres cualesquiera y otra `A`. De `ABA`, `ABBA` y `ABBBA`, solo `ABBA` cumple (`ABBA`, además, es el nombre de un grupo musical sueco).

In [33]:
print(re.search('A..A', 'ABA'))      # no se ajusta a la condición
print(re.search('A..A', 'ABBA'))     # se ajusta a la condición
print(re.search('A..A', 'ABBBA'))    # no se ajusta a la condición

None
<re.Match object; span=(0, 4), match='ABBA'>
None


**El asterisco `*`** (PDF pág. 67): `'AB*'` coincide con una `A` seguida de **cero o más** `B`.

In [34]:
print(re.search('AB*', 'A'))          # se ajusta: A con cero B
print(re.search('AB*', 'AA'))         # se ajusta: la primera A
print(re.search('AB*', 'J-HOP'))      # no se ajusta: no hay A
print(re.search('AB*', 'X-MAN'))      # se ajusta: la A de MAN
print(re.search('AB*', 'CABBA'))      # se ajusta: ABB
print(re.search('AB*', 'CABBBBBA'))   # se ajusta: ABBBBB

<re.Match object; span=(0, 1), match='A'>
<re.Match object; span=(0, 1), match='A'>
None
<re.Match object; span=(3, 4), match='A'>
<re.Match object; span=(1, 4), match='ABB'>
<re.Match object; span=(1, 7), match='ABBBBB'>


**Nota sobre el PDF:** el texto de la pág. 67 dice "El asterisco (x)"; es un error tipográfico, el metacarácter es `*`.

**La interrogación `?`** (PDF pág. 68): `'AB?'` coincide con una `A` seguida de **cero o una** `B`. El resultado parece similar al de `*`, pero con `CABBBBBA` y `CABBA` la coincidencia es solo `AB`, porque `?` admite como máximo una repetición.

In [35]:
print(re.search('AB?', 'A'))          # se ajusta: A
print(re.search('AB?', 'AA'))         # se ajusta: A
print(re.search('AB?', 'J-HOP'))      # no se ajusta
print(re.search('AB?', 'X-MAN'))      # se ajusta: A
print(re.search('AB?', 'CABBA'))      # se ajusta: AB (solo una B)
print(re.search('AB?', 'CABBBBBA'))   # se ajusta: AB (solo una B)

<re.Match object; span=(0, 1), match='A'>
<re.Match object; span=(0, 1), match='A'>
None
<re.Match object; span=(3, 4), match='A'>
<re.Match object; span=(1, 3), match='AB'>
<re.Match object; span=(1, 3), match='AB'>


**El signo más `+`** (PDF pág. 69): `'AB+'` coincide con una `A` seguida de **una o más** `B`. No coincide con `A` sola, pero sí con `AB`, `ABB`, `ABBBB`, etc.

In [36]:
print(re.search('AB+', 'A'))          # no se ajusta: falta al menos una B
print(re.search('AB+', 'AA'))         # no se ajusta
print(re.search('AB+', 'J-HOP'))      # no se ajusta
print(re.search('AB+', 'X-MAN'))      # no se ajusta
print(re.search('AB+', 'CABBA'))      # se ajusta: ABB
print(re.search('AB+', 'CABBBBBA'))   # se ajusta: ABBBBB

None
None
None
None
<re.Match object; span=(1, 4), match='ABB'>
<re.Match object; span=(1, 7), match='ABBBBB'>


**Nota sobre el PDF:** el resumen de metacaracteres de la pág. 65 no incluye `?`, pero las págs. 66–68 lo usan; se incluye aquí junto con `*` y `+` por completitud. En la diapositiva de `?` el texto lo llama "carácter ampliamente utilizado" (traducción confusa de "carácter muy usado").

### 11.6 `re.findall` (PDF pág. 70)

`re.findall(patron, cadena)` extrae **todas** las coincidencias del patrón y las devuelve en una lista (en lugar de solo la primera, como `search`).

In [37]:
txt3 = 'My life my life my life in the sunshine'
print(re.findall('[Mm]y', txt3))

['My', 'my', 'my']


**Clave:**
- `re.search` devuelve la **primera** coincidencia (o `None`); `re.findall` devuelve **todas**, en una lista.
- `.` = cualquier carácter; `*` = cero o más; `+` = una o más; `?` = cero o una (ampliación); `^` y `$` anclan al inicio y al final; `[]` define un conjunto; `|` define alternativas.

**Error común:** usar el resultado de `re.search` sin comprobar si es `None`: `re.search(...).group()` falla con `AttributeError` cuando no hay coincidencia.

**Para el quiz:** el metacarácter `^` coincide con el inicio de la cadena y `$` con el final.

**Referencia:** PDF parte 2, págs. 56–70

## Proyecto de la unidad

### Análisis de la urbanización

**Problema (PDF págs. 12–15).** Una nación de Asia tiene cuatro grandes ciudades. Se debe crear un diccionario llamado `city_pop` con el nombre y la población (en miles) de A, B, C y D:

- A = 9765
- B = 3441
- C = 2954
- D = 1531

Y una función `urban` que imprima la población de la ciudad con **mayor** población, la de la ciudad con **menor** población, la **diferencia** entre ambas y el **valor medio** de la población de las cuatro ciudades.

**Plan de programación (pseudocódigo, PDF pág. 14):**

1. Inicio.
2. Almacenar los nombres y la población de las ciudades en el diccionario `city_pop`.
3. Inicializar `max_pop`, `min_pop`, `pop_sum` y el valor `n`.
4. Para cada ciudad del diccionario:
5. Si la población de la ciudad es mayor que `max_pop`:
6. Guardar esa población como la máxima.
7. Si la población de la ciudad es menor que `min_pop`:
8. Guardar esa población como la mínima.
9. Acumular la población en `pop_sum` y contar la ciudad en `n`.
10. Imprimir la población máxima, la mínima, la diferencia y la media.
11. Fin.

El diagrama de flujo del PDF sigue los mismos pasos: leer el diccionario, inicializar las variables y, en un ciclo `for nombre, pop in city_pop`, comparar con el máximo y el mínimo y acumular la suma y el contador.

**Código final (PDF págs. 13 y 15), con comentarios.** Observa cómo se combinan `global` (para `max_pop`, `min_pop` y `pop_sum`, que se inicializan **fuera** de la función) y una variable local (`n`, que nace dentro de la función).

In [38]:
def urban(city):
    global max_pop, min_pop, pop_sum     # estas tres son globales: se modifican desde la función
    n = 0                                # n es local: se crea en cada llamada
    for name, pop in city.items():       # obtiene el valor máximo y mínimo iterando en el ciclo
        if pop > max_pop:
            max_pop = pop
        if pop < min_pop:
            min_pop = pop
        pop_sum += pop
        n += 1
    print('población máxima: ', max_pop)
    print('población mínima: ', min_pop)
    print('diferencia entre población máxima y población mínima:', max_pop - min_pop)
    print('población media:', pop_sum / n)


max_pop = 0
min_pop = 1000000
pop_sum = 0
# almacena los datos estadísticos de población (unidad: miles) en un diccionario
city_pop = {'A': 9765, 'B': 3441, 'C': 2954, 'D': 1531}
urban(city_pop)

población máxima:  9765
población mínima:  1531
diferencia entre población máxima y población mínima: 8234
población media: 4422.75


**Resultado esperado:** población máxima 9765, mínima 1531, diferencia 8234 y media 4422.75.

**Nota sobre el PDF:** (1) la salida de la diapositiva usa etiquetas con mayúscula y ` : ` ("Población máxima : 9765") y una coma decimal ("4422,75"), que no coinciden con el código: el código imprime con minúsculas y Python escribe el decimal con punto (`4422.75`). (2) El diagrama de flujo usa `pop_suma`, mientras que el código usa `pop_sum`; el nombre correcto es `pop_sum`. (3) En el código, el comentario de la línea 19 se superpone con otras líneas por la maquetación; no afecta a la ejecución.

**Ampliación (no está en el PDF): la trampa de las variables globales.** Este diseño funciona, pero `pop_sum` es global y **conserva su valor entre llamadas**. Si se vuelve a llamar a `urban` sin reiniciar las globales, el acumulado crece mientras `n` (local) vuelve a empezar, y la media sale mal. Es exactamente el tipo de efecto secundario que las clausuras evitan.

In [39]:
print("Segunda llamada SIN reiniciar las globales:")
urban(city_pop)          # pop_sum ya traía el total de la llamada anterior

print()
print("Tercera llamada reiniciando las globales:")
max_pop = 0
min_pop = 1000000
pop_sum = 0
urban(city_pop)

Segunda llamada SIN reiniciar las globales:
población máxima:  9765
población mínima:  1531
diferencia entre población máxima y población mínima: 8234
población media: 8845.5

Tercera llamada reiniciando las globales:
población máxima:  9765
población mínima:  1531
diferencia entre población máxima y población mínima: 8234
población media: 4422.75


En la segunda llamada la media sale `8845.5` (el doble) porque `pop_sum` quedó con 17691 de la llamada anterior y se le sumaron otros 17691, pero `n` volvió a 4. Reiniciando las globales, el resultado vuelve a ser correcto.

**Ampliación (no está en el PDF): versión sin variables globales.** La misma lógica con variables locales y un valor de retorno. No depende de variables externas, se puede llamar las veces que se quiera y es más fácil de probar (es la base del ejercicio 8 de la práctica).

In [40]:
def urban_sin_globales(city):
    max_pop = None
    min_pop = None
    pop_sum = 0
    n = 0
    for name, pop in city.items():
        if max_pop is None or pop > max_pop:
            max_pop = pop
        if min_pop is None or pop < min_pop:
            min_pop = pop
        pop_sum += pop
        n += 1
    return max_pop, min_pop, max_pop - min_pop, pop_sum / n


resultado = urban_sin_globales(city_pop)
print("máxima, mínima, diferencia, media =", resultado)
print("segunda llamada:", urban_sin_globales(city_pop))

máxima, mínima, diferencia, media = (9765, 1531, 8234, 4422.75)
segunda llamada: (9765, 1531, 8234, 4422.75)


**Referencia:** PDF parte 2, págs. 9–15

### Programación en pareja

La unidad cierra con tres diapositivas de "práctica de programación en pareja" (págs. 72–73) y un ejercicio (pág. 74). Ideas del PDF, dirigidas al docente:

- **Lineamientos, mecanismos y plan de contingencia:** establecer pautas para que los alumnos formen parejas y se mantengan emparejados (por ejemplo, turnarse para manejar el mouse). Prever qué hacer si uno de los socios se ausenta o decide no participar, aclarando que el estudiante activo **no será sancionado** porque la pareja no haya funcionado.
- **Emparejar con habilidades similares**, no necesariamente iguales: el emparejamiento muy desigual suele producir una participación desequilibrada. La programación en pareja **no es "divide y vencerás"**, sino un esfuerzo de colaboración en todo el proyecto; evitar emparejar a un estudiante muy débil con uno muy fuerte.
- **Motivar con incentivos extra**, especialmente con estudiantes avanzados; puede ser útil emparejar solo para una o dos tareas.
- **Evitar trampas en la colaboración:** evaluar resultados individuales sin perder los beneficios de colaborar; revisar el diseño y la evaluación del curso, discutir explícitamente qué conductas se considerarán trampa y hacer tareas significativas.
- **Entorno de aprendizaje colaborativo:** el instructor exige que los alumnos trabajen juntos (parejas en tareas de programación, grupos pequeños que discuten una pregunta, estudio conjunto fuera de clase). Es distinto de dividir el trabajo, donde cada uno resuelve solo una parte.

**Ejercicio en pareja (pág. 74):** dada una lista `lst` con los valores de 1 a 100, extraer una lista resultado con los elementos divisibles por 5 o por 7. Se declara una función `func1(a)` que anida las funciones `func2` y `func3`; `func1` llama a ambas e imprime los números divisibles por 5 o 7, ordenados con `sorted()`. Se resuelve en el **ejercicio 7** de la práctica.

**Nota sobre el PDF:** en la diapositiva `func2` y `func3` aparecen sin sangría (parecen definidas fuera de `func1`) y la lista resultado de ejemplo repite los múltiplos de 35 (`35, 35` y `70, 70`), porque se concatenan las dos listas sin quitar duplicados. Esa salida de ejemplo es coherente con el enunciado literal.

**Referencia:** PDF parte 2, págs. 71–74

## Práctica

Los ejercicios 1 a 3 y 7 provienen de los ejercicios del PDF; los demás son propios (**Ampliación (no está en el PDF):**). Las celdas con `pass` imprimen "Incorrecto" hasta que escribas tu solución; es lo esperado.

### Ejercicio 1 — Función anidada `saludos` (PDF Q1, pág. 47)

Define la función `saludos()`. Dentro debe definir la función anidada `decir_hi()` que devuelve la cadena `'hello'`; `saludos` llama a `decir_hi` y **devuelve** su resultado.

(El PDF imprime el saludo; aquí se devuelve para poder verificarlo.)

In [41]:
def saludos():
    # Escribe tu solución aquí
    pass

In [42]:
verificar("saludos() devuelve 'hello'", saludos(), 'hello')
verificar("el resultado es una cadena", isinstance(saludos(), str), True)

Incorrecto: saludos() devuelve 'hello'. Se esperaba 'hello' y se obtuvo None
Incorrecto: el resultado es una cadena. Se esperaba True y se obtuvo False


<details><summary>Ver solución</summary>

```python
def saludos():
    def decir_hi():
        return 'hello'

    return decir_hi()
```

`decir_hi` solo existe dentro de `saludos`; se la llama dentro de la función externa y su valor se devuelve hacia afuera.
</details>

### Ejercicio 2 — Clausura `calc` (PDF Q2, pág. 48)

Escribe `calc()` con `a = 3`, `b = 5` y la función interna `mul_add(x)` que devuelve `a * x + b`; `calc` devuelve `mul_add`. Luego `num = calc()` y `num(3)` debe dar `14`.

In [43]:
def calc():
    # Escribe tu solución aquí
    pass

In [44]:
num = calc()
verificar("num(3)", num(3) if callable(num) else None, 14)
verificar("num(0)", num(0) if callable(num) else None, 5)
verificar("num(1)", num(1) if callable(num) else None, 8)

Incorrecto: num(3). Se esperaba 14 y se obtuvo None
Incorrecto: num(0). Se esperaba 5 y se obtuvo None
Incorrecto: num(1). Se esperaba 8 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def calc():
    a = 3
    b = 5

    def mul_add(x):
        return a * x + b

    return mul_add
```

`calc` devuelve la función interna sin llamarla (sin paréntesis). `mul_add` conserva `a` y `b`: es una clausura. `3*3+5 = 14`.
</details>

### Ejercicio 3 — Clausura con lambda (PDF Q3, pág. 49)

Escribe `calc_lambda()` con `a = 3` y `b = 5` que devuelva **una expresión lambda** equivalente a `mul_add`. `calc_lambda()(3)` debe dar `14`.

In [45]:
def calc_lambda():
    # Escribe tu solución aquí
    pass

In [46]:
f = calc_lambda()
verificar("f(3)", f(3) if callable(f) else None, 14)
verificar("f(2)", f(2) if callable(f) else None, 11)
verificar("f(0)", f(0) if callable(f) else None, 5)

Incorrecto: f(3). Se esperaba 14 y se obtuvo None
Incorrecto: f(2). Se esperaba 11 y se obtuvo None
Incorrecto: f(0). Se esperaba 5 y se obtuvo None


<details><summary>Ver solución</summary>

```python
def calc_lambda():
    a = 3
    b = 5
    return lambda x: a * x + b
```

La lambda usa `a` y `b` de la función externa, así que además es una clausura.
</details>

### Ejercicio 4 — Contador global con `global` (ampliación)

Existe una variable global `contador_global`. Escribe `incrementar()` que sume 1 a esa variable **global** (usa `global`). No debe devolver nada.

In [47]:
contador_global = 0


def incrementar():
    # Escribe tu solución aquí
    pass

In [48]:
contador_global = 0
incrementar()
incrementar()
verificar("tras dos llamadas", contador_global, 2)
incrementar()
verificar("tras tres llamadas", contador_global, 3)

Incorrecto: tras dos llamadas. Se esperaba 2 y se obtuvo 0
Incorrecto: tras tres llamadas. Se esperaba 3 y se obtuvo 0


<details><summary>Ver solución</summary>

```python
contador_global = 0


def incrementar():
    global contador_global
    contador_global += 1
```

Sin `global`, `contador_global += 1` lanzaría `UnboundLocalError`, porque Python trataría el nombre como local.
</details>

### Ejercicio 5 — Contadores independientes con `nonlocal` (ampliación)

Escribe `make_contador(inicio=0)` que devuelva una función. Cada vez que se llame a esa función, aumenta en 1 un contador privado (que parte de `inicio`) y devuelve el nuevo valor. Dos contadores creados por separado deben ser independientes.

In [49]:
def make_contador(inicio=0):
    # Escribe tu solución aquí
    pass

In [50]:
def llamar(funcion):
    return funcion() if callable(funcion) else None


c1 = make_contador()
c2 = make_contador(10)
valores = [llamar(c1), llamar(c1), llamar(c2), llamar(c1)]
verificar("secuencia de llamadas alternadas", valores, [1, 2, 11, 3])

Incorrecto: secuencia de llamadas alternadas. Se esperaba [1, 2, 11, 3] y se obtuvo [None, None, None, None]


<details><summary>Ver solución</summary>

```python
def make_contador(inicio=0):
    cuenta = inicio

    def contador():
        nonlocal cuenta
        cuenta += 1
        return cuenta

    return contador
```

`cuenta` es una variable local de `make_contador` capturada por la clausura. Se usa `nonlocal` para poder **asignarla**; cada llamada a `make_contador` crea su propia `cuenta`.
</details>

### Ejercicio 6 — Funciones de primera clase en una lista (ampliación)

Escribe `aplicar_todas(funciones, x)` que reciba una lista de funciones y un valor, y devuelva la lista con el resultado de aplicar cada función a `x`, en el mismo orden.

In [51]:
def aplicar_todas(funciones, x):
    # Escribe tu solución aquí
    pass

In [52]:
verificar("abs, float y doble", aplicar_todas([abs, float, lambda v: v * 2], -3), [3, -3.0, -6])
verificar("lista vacía", aplicar_todas([], 5), [])
verificar("métodos de str y len", aplicar_todas([str.upper, len], "hola"), ["HOLA", 4])

Incorrecto: abs, float y doble. Se esperaba [3, -3.0, -6] y se obtuvo None
Incorrecto: lista vacía. Se esperaba [] y se obtuvo None
Incorrecto: métodos de str y len. Se esperaba ['HOLA', 4] y se obtuvo None


<details><summary>Ver solución</summary>

```python
def aplicar_todas(funciones, x):
    resultados = []
    for funcion in funciones:
        resultados.append(funcion(x))
    return resultados
```

Las funciones son objetos: se pueden guardar en una lista, recorrer y llamar con paréntesis. Equivale a `[f(x) for f in funciones]`.
</details>

### Ejercicio 7 — Múltiplos de 5 o de 7 con funciones anidadas (PDF programación en pareja, pág. 74)

Escribe `func1(a)` que reciba una lista de números. Dentro debe definir las funciones anidadas `func2()` (múltiplos de 5 de `a`) y `func3()` (múltiplos de 7 de `a`); `func1` las llama a ambas y devuelve la unión de los resultados **ordenada** con `sorted()`. Igual que en el PDF, no se eliminan los duplicados (un múltiplo de 35 aparece dos veces).

In [53]:
def func1(a):
    # Escribe tu solución aquí
    pass

In [54]:
verificar("1 a 40", func1(list(range(1, 41))), [5, 7, 10, 14, 15, 20, 21, 25, 28, 30, 35, 35, 40])
verificar("lista vacía", func1([]), [])
verificar("sin múltiplos", func1([1, 2, 3]), [])

Incorrecto: 1 a 40. Se esperaba [5, 7, 10, 14, 15, 20, 21, 25, 28, 30, 35, 35, 40] y se obtuvo None
Incorrecto: lista vacía. Se esperaba [] y se obtuvo None
Incorrecto: sin múltiplos. Se esperaba [] y se obtuvo None


<details><summary>Ver solución</summary>

```python
def func1(a):
    def func2():
        return [i for i in a if i % 5 == 0]

    def func3():
        return [i for i in a if i % 7 == 0]

    return sorted(func2() + func3())
```

`func2` y `func3` leen `a` del ámbito envolvente (E), sin recibirla como parámetro. El 35 es múltiplo de 5 y de 7, por eso aparece dos veces.
</details>

### Ejercicio 8 — Análisis de la urbanización sin globales (ampliación basada en el proyecto)

Escribe `analizar_ciudades(city_pop)` que reciba un diccionario `{nombre: población}` y devuelva la tupla `(máxima, mínima, diferencia, media)` **sin usar variables globales**.

In [55]:
def analizar_ciudades(city_pop):
    # Escribe tu solución aquí
    pass

In [56]:
verificar("datos del PDF", analizar_ciudades({'A': 9765, 'B': 3441, 'C': 2954, 'D': 1531}), (9765, 1531, 8234, 4422.75))
verificar("dos ciudades", analizar_ciudades({'X': 10, 'Y': 20}), (20, 10, 10, 15.0))
verificar("una ciudad", analizar_ciudades({'Z': 7}), (7, 7, 0, 7.0))

Incorrecto: datos del PDF. Se esperaba (9765, 1531, 8234, 4422.75) y se obtuvo None
Incorrecto: dos ciudades. Se esperaba (20, 10, 10, 15.0) y se obtuvo None
Incorrecto: una ciudad. Se esperaba (7, 7, 0, 7.0) y se obtuvo None


<details><summary>Ver solución</summary>

```python
def analizar_ciudades(city_pop):
    poblaciones = list(city_pop.values())
    maxima = max(poblaciones)
    minima = min(poblaciones)
    return maxima, minima, maxima - minima, sum(poblaciones) / len(poblaciones)
```

Todo el estado vive en variables locales, así que la función se puede llamar varias veces sin que los resultados se contaminen entre sí.
</details>

## Apuntes para estudiar

### Clave

- Regla LEGB: el nombre se busca en Local, luego Enclosing (función envolvente), luego Global y por último Built-in.
- Una variable local nace al llamar a la función y desaparece al terminar. Una local y una global con el mismo nombre ocupan memoria distinta.
- `global x` hace que la función use (y modifique) la variable `x` del módulo.
- `nonlocal x` enlaza con la variable `x` del espacio de nombres envolvente más cercano (vinculación); exige que exista en una función exterior.
- Las funciones son objetos de primera clase: se pasan, se asignan, se devuelven y se guardan en listas o diccionarios. Las que reciben o devuelven funciones son de orden superior.
- Una función anidada puede leer las variables de su función padre.
- Clausura: función anidada devuelta que conserva las variables (libres) de la función externa. Sirve para ocultar datos y limitar las globales; cada clausura tiene su propio estado.
- Una lambda devuelta por una función que usa sus variables también es una clausura.
- `re.search` devuelve la primera coincidencia o `None`; `re.findall` devuelve todas. Metacaracteres básicos: `^`, `$`, `.`, `*`, `+`, `[]`, `|`.

### Errores comunes

- Creer que asignar dentro de una función cambia la variable global (crea una local).
- Nombrar variables con nombres incorporados (`abs`, `list`, `sum`...) y bloquear la función original.
- `UnboundLocalError`: leer y asignar una variable sin declararla `global` o `nonlocal`.
- Usar `global` para una variable de la función exterior (`NameError`) o `nonlocal` para una variable solo global (`SyntaxError`).
- Pasar `f()` en vez de `f` cuando se quiere pasar la función; devolver `mul_add()` en vez de `mul_add` al crear una clausura.
- Olvidar que los resultados de `re.search` pueden ser `None` antes de llamar a `group()`.
- Depender de variables globales que conservan su valor entre llamadas (el acumulado de `urban` se contamina).

### Para el quiz

- Orden de la regla de ámbito: L, E, G, B.
- La variable local desaparece al finalizar la función; la palabra clave `global` usa la variable del módulo.
- `nonlocal` busca la variable del espacio de nombres envolvente más cercano.
- Función anidada = función definida dentro de otra; su razón principal de uso es entender las clausuras.
- Usos principales de la clausura: limitar el uso de variables globales y ocultar datos.
- `c1 = makecounter(); c2 = makecounter()` producen contadores con memoria independiente (`c1 1`, `c1 2`, `c2 1`).
- `nonlocal` también modifica la variable local de una clausura (`5 12 21` en el ejemplo del acumulador).
- `^` = inicio de cadena, `$` = fin de cadena, `.` = un carácter cualquiera, `*` = cero o más repeticiones, `+` = una o más.

## Autoevaluación

**1. ¿En qué orden busca Python un nombre según la regla LEGB y qué representa cada letra?**

<details><summary>Ver respuesta</summary>

Primero en el ámbito Local (la función actual), luego en el Enclosing (funciones envolventes), después en el Global (módulo) y por último en el Built-in (nombres incorporados). Se detiene en el primer ámbito donde encuentra el nombre.
</details>

**2. En el siguiente código, ¿qué imprime la última línea y por qué?**

```python
n = 100
def f():
    n = 200
f()
print(n)
```

<details><summary>Ver respuesta</summary>

Imprime `100`. La asignación `n = 200` dentro de `f` crea una variable local nueva con memoria separada; la variable global `n` no se modifica. Solo cambiaría con `global n` dentro de `f`.
</details>

**3. ¿Cuál es la diferencia entre `global` y `nonlocal`?**

<details><summary>Ver respuesta</summary>

`global` enlaza con una variable del espacio de nombres del módulo. `nonlocal` enlaza con la variable del espacio de nombres envolvente más cercano, que debe estar en una función exterior. Usar `nonlocal` con una variable que solo es global produce `SyntaxError`, y usar `global` para una variable de la función exterior produce `NameError`.
</details>

**4. ¿Qué es una clausura y para qué sirve?**

<details><summary>Ver respuesta</summary>

Es una función anidada que se devuelve desde la función externa y que conserva las variables de esa función externa aunque esta ya haya terminado. Sirve para limitar el uso de variables globales y para ocultar datos; además cada clausura creada tiene su propio estado independiente.
</details>

**5. ¿Qué imprime este código y por qué hace falta `nonlocal`?**

```python
def makecounter():
    count = 0
    def counter():
        nonlocal count
        count += 1
        return count
    return counter

c1 = makecounter()
c2 = makecounter()
print(c1(), c1(), c2())
```

<details><summary>Ver respuesta</summary>

Imprime `1 2 1`. `c1` y `c2` tienen cada una su propia variable `count`. `nonlocal` es necesario porque `counter` **asigna** `count`, que pertenece a `makecounter`; sin esa declaración Python trataría `count` como local y lanzaría `UnboundLocalError`.
</details>

---

[Índice de la semana](../semana-02.ipynb) | [Anterior](../clase-1-funciones-recursividad-lambda/u19-lambda-filter-map-reduce.ipynb) | [Siguiente](u21-clases-y-objetos.ipynb)